In [2]:
import pandas as pd
import os
import re
import time
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

# CONFIG
RAW_FILE = r"C:\Users\23225632\Documents\Projects\UserLookup\data\Kenya Offrole & CWK Dump_27 FEB.xlsx"
SHEET_NAME = "Sheet1"
HOSTNAME_COL = "Hostname"
OUTPUT_FILE = "output.xlsx"
OUTPUT_SHEET = "user data"

# Thread pool sizing. 20–40 is usually fine for domain calls.
MAX_WORKERS = 32
# Subprocess timeout per call (seconds)
NETUSER_TIMEOUT = 8
# Retries if a call times out or fails
RETRIES = 2
VERBOSE = True  
# END CONFIG

In [ ]:
def log(msg):
    if VERBOSE:
        print(msg)

# -------------------- Load & Normalize --------------------------
df = pd.read_excel(RAW_FILE, sheet_name=SHEET_NAME, engine="openpyxl")

# Split multi-line hostnames into separate rows
df = df.assign(**{
    HOSTNAME_COL: df[HOSTNAME_COL].astype(str).str.split(r'[\n\r]+')
})
df = df.explode(HOSTNAME_COL).reset_index(drop=True)

# Extract AUUID (first sequence of 5+ digits) from Hostname
df["AUUID_digits"] = df[HOSTNAME_COL].astype(str).str.extract(r"(\d{5,})", expand=False)

# Only query valid (non-null) IDs
mask_valid = df["AUUID_digits"].notna()
unique_ids = pd.unique(df.loc[mask_valid, "AUUID_digits"].astype(str))

log(f"Total rows: {len(df)} | Unique AUUIDs to query: {len(unique_ids)}")

# -------------------- Load Existing Staff Names -----------------
existing_mapping = {}
if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_excel(OUTPUT_FILE, sheet_name=OUTPUT_SHEET, engine="openpyxl")
        # Map AUUID_digits to Staff Name if available
        for _, row in existing_df.iterrows():
            auuid = str(row["AUUID_digits"])
            staff_name = row.get("Staff Name", None)
            if pd.notna(staff_name):
                existing_mapping[auuid] = staff_name
    except Exception as e:
        log(f"[LOAD EXISTING ERROR] {e}")

# -------------------- Domain Query Function with Retries --------
def extract_staff_name(auuid, timeout=NETUSER_TIMEOUT):
    auuid = str(auuid).strip()
    if os.name != "nt" or not auuid or auuid == 'nan':
        return None
    cmd = ["net", "user", "/domain", auuid]
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            shell=False
        )
    except subprocess.TimeoutExpired:
        log(f"[TIMEOUT] {auuid}")
        return None
    except Exception as e:
        log(f"[ERROR] {auuid}: {e}")
        return None
    if result.returncode != 0:
        if result.stderr:
            log(f"[RC{result.returncode}] {auuid} stderr: {result.stderr.strip()}")
        else:
            log(f"[RC{result.returncode}] {auuid} stdout: {result.stdout.strip()[:120]}...")
        return None
    for line in result.stdout.splitlines():
        m = FULLNAME_REGEX.match(line)
        if m:
            return m.group(1).strip() or None
    for line in result.stdout.splitlines():
        if "Full" in line and "Name" in line and ":" in line:
            try:
                return line.split(":", 1)[1].strip() or None
            except Exception:
                pass
    return None

def extract_with_retries(auuid):
    for attempt in range(RETRIES + 1):
        name = extract_staff_name(auuid)
        if name:
            return name
        time.sleep(0.5)
    return None

# -------------------- Parallel Execution ------------------------
mapping = existing_mapping.copy()
ids_to_lookup = [uid for uid in unique_ids if uid not in mapping or not mapping[uid]]

if ids_to_lookup:
    log(f"Starting parallel lookups with {MAX_WORKERS} workers for {len(ids_to_lookup)} IDs...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_with_retries, uid): uid for uid in ids_to_lookup}
        done_count = 0
        for future in as_completed(futures):
            uid = futures[future]
            try:
                mapping[uid] = future.result()
            except Exception as e:
                mapping[uid] = None
                log(f"[FUTURE ERROR] {uid}: {e}")
            done_count += 1
            if done_count % 50 == 0 or done_count == len(ids_to_lookup):
                log(f"Completed {done_count}/{len(ids_to_lookup)} lookups")

log(f"Resolved {sum(v is not None for v in mapping.values())} / {len(unique_ids)} AUUIDs")

# Map back to DataFrame
df["Staff Name"] = df["AUUID_digits"].astype(str).map(mapping)

# -------------------- Save Output -------------------------------
out_cols = [HOSTNAME_COL, "AUUID_digits", "Staff Name"]

if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_excel(OUTPUT_FILE, sheet_name=OUTPUT_SHEET, engine="openpyxl")
        combined_df = pd.concat([existing_df, df[out_cols]], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=[HOSTNAME_COL, "AUUID_digits"])
        combined_df.to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)
        log(f"Appended and wrote {len(combined_df)} rows to {OUTPUT_FILE} ({OUTPUT_SHEET}).")
    except Exception as e:
        log(f"[SAVE ERROR] {e}")
else:
    df[out_cols].to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)
    log(f"Done. Wrote {len(df)} rows to {OUTPUT_FILE} ({OUTPUT_SHEET}).")


Total rows: 701 | Unique AUUIDs to query: 624
Starting parallel lookups with 32 workers...
Completed 50/624 lookups
Completed 100/624 lookups
Completed 150/624 lookups
Completed 200/624 lookups
Completed 250/624 lookups
Completed 300/624 lookups
Completed 350/624 lookups
Completed 400/624 lookups
Completed 450/624 lookups
Completed 500/624 lookups
Completed 550/624 lookups
Completed 600/624 lookups
Completed 624/624 lookups
Resolved 0 / 624 AUUIDs
Appended and wrote 1384 rows to output.xlsx (user data).


In [ ]:
import pandas as pd
import os
import subprocess
RAW_FILE = r"C:\Users\23225632\Documents\Projects\UserLookup\data\Kenya Offrole & CWK Dump_27 FEB.xlsx"
SHEET_NAME = "Sheet1"
HOSTNAME_COL = "Hostname"
OUTPUT_FILE = "output.xlsx"
OUTPUT_SHEET = "user data"
# Load and normalize data
df = pd.read_excel(RAW_FILE, sheet_name=SHEET_NAME, engine="openpyxl")
df = df.assign(**{HOSTNAME_COL: df[HOSTNAME_COL].astype(str).str.split(r'[\n\r]+')})
df = df.explode(HOSTNAME_COL).reset_index(drop=True)
df["AUUID_digits"] = df[HOSTNAME_COL].astype(str).str.extract(r"(\d{5,})")[0]
# Extract staff name using net user /domain
def extract_staff_name(auuid):
    auuid = str(auuid).strip()
    if not auuid or auuid == 'nan':
        return None
    command = f'net user /domain "{auuid}"'
    try:
        result = subprocess.run(command, capture_output=True, text=True, shell=True, timeout=8)
    except subprocess.TimeoutExpired:
        return None
    except Exception:
        return None
    for line in result.stdout.splitlines():
        if 'Full Name' in line and ':' in line:
            parts = line.split(':', 1)
            if len(parts) > 1:
                return parts[1].strip()
    return None
# Apply extraction and update output file
df["Staff Name"] = df["AUUID_digits"].apply(extract_staff_name)
out_cols = [HOSTNAME_COL, "AUUID_digits", "Staff Name"]
if os.path.exists(OUTPUT_FILE):
    existing_df = pd.read_excel(OUTPUT_FILE, sheet_name=OUTPUT_SHEET, engine="openpyxl")
    combined_df = pd.concat([existing_df, df[out_cols]], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=[HOSTNAME_COL, "AUUID_digits"])
    combined_df.to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)
else:
    df[out_cols].to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)